# MİHENK — bounded A100 participant run

Run the application and inference together inside this active notebook. Upload the supplied source ZIP; everything listens on loopback. The first run uses five independent accounts and the same application commands as the website. No participant role or objective is supplied. Optional decision notes are one-sentence summaries.

Candidate: Qwen3.5-9B, BF16, thinking off, 16K serving context, 4K complete input and 256 output tokens per decision. These are starting settings, not measured throughput. This notebook has not yet been executed on the user's GPU.

References: [official Qwen card](https://huggingface.co/Qwen/Qwen3.5-9B), [vLLM 0.21.0 tool parser source](https://github.com/vllm-project/vllm/blob/v0.21.0/docs/features/tool_calling.md), [tokenizer request contract](https://github.com/vllm-project/vllm/blob/v0.21.0/vllm/entrypoints/serve/tokenize/protocol.py), [Colab resource limits](https://research.google.com/colaboratory/faq.html). This pinned vLLM release names the XML tool parser qwen3_xml.

In [ ]:
from google.colab import files
from pathlib import Path
import json, os, sys, subprocess, urllib.request, hashlib, tarfile, zipfile, time, shutil

root = Path("/content/mihenk")
root.mkdir(exist_ok=True)
uploaded = files.upload()
archive_name = next(name for name in uploaded if name.endswith(".zip"))
with zipfile.ZipFile(archive_name) as archive:
    for item in archive.infolist():
        target = (root / item.filename).resolve()
        if not target.is_relative_to(root.resolve()):
            raise ValueError("Archive contains an invalid path")
    archive.extractall(root)
assert (root / "server/http.mjs").exists(), "Upload mihenk-colab.zip"
os.chdir(root)
runtime_dir = root / ".rehearsal"
runtime_dir.mkdir(exist_ok=True)
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], text=True)
print(gpu)
assert "A100" in gpu, "This starting configuration is for an A100. Select that runtime before continuing."


In [ ]:
NODE_VERSION = "24.18.0"
node_dir = Path("/content/node-runtime")
node_dir.mkdir(exist_ok=True)
archive = node_dir / f"node-v{NODE_VERSION}-linux-x64.tar.xz"
base = f"https://nodejs.org/dist/v{NODE_VERSION}/"
if not archive.exists():
    urllib.request.urlretrieve(base + archive.name, archive)
checksums = urllib.request.urlopen(base + "SHASUMS256.txt").read().decode()
expected = next(line.split()[0] for line in checksums.splitlines() if line.split()[-1] == archive.name)
assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected
with tarfile.open(archive) as bundle:
    bundle.extractall(node_dir, filter="data")
node_bin = node_dir / f"node-v{NODE_VERSION}-linux-x64/bin"
os.environ["PATH"] = str(node_bin) + os.pathsep + os.environ["PATH"]
subprocess.check_call(["node", "--version"])
subprocess.check_call(["npm", "ci"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "vllm==0.21.0"])


In [ ]:
from huggingface_hub import HfApi
MODEL = "Qwen/Qwen3.5-9B"
lock_path = runtime_dir / "runtime-lock.json"
if lock_path.exists():
    runtime = json.loads(lock_path.read_text())
    assert runtime["model"] == MODEL
else:
    runtime = {
        "model": MODEL,
        "revision": HfApi().model_info(MODEL).sha,
        "node": NODE_VERSION,
        "vllm": "0.21.0",
        "gpu": gpu,
        "dtype": "bfloat16",
        "context": 16384,
        "thinking": False,
        "speculative_decoding": False,
        "tool_parser": "qwen3_xml",
    }
    lock_path.write_text(json.dumps(runtime, indent=2))
(runtime_dir / "pip-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print(json.dumps(runtime, indent=2))


In [ ]:
app_log = open(runtime_dir / "application.log", "a")
model_log = open(runtime_dir / "inference.log", "a")
application = subprocess.Popen(["node", "tools/serve-rehearsal.mjs"], cwd=root, stdout=app_log, stderr=subprocess.STDOUT)
inference = subprocess.Popen([
    "vllm", "serve", MODEL, "--revision", runtime["revision"],
    "--host", "127.0.0.1", "--port", "8000",
    "--tensor-parallel-size", "1", "--dtype", "bfloat16",
    "--max-model-len", "16384", "--max-num-seqs", "4",
    "--gpu-memory-utilization", "0.85", "--enable-chunked-prefill",
    "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_xml",
    "--reasoning-parser", "qwen3",
], stdout=model_log, stderr=subprocess.STDOUT)
print("Loading the pinned model. Progress is in .rehearsal/inference.log.")


In [ ]:
deadline = time.monotonic() + 20 * 60
while time.monotonic() < deadline:
    if application.poll() is not None or inference.poll() is not None:
        raise RuntimeError("A process stopped. Inspect application.log and inference.log.")
    try:
        models = json.load(urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2))
        if models.get("data"):
            break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError("Model loading exceeded 20 minutes; inspect inference.log.")
config = {
    "engine": "model", "mode": "tool", "participants": 5,
    "inferenceConcurrency": 2, "maxDecisionsPerParticipant": 20,
    "maxInputTokensPerDecision": 4096, "maxOutputTokensPerDecision": 256,
    "maxTotalTokens": 450000, "requestTimeoutSeconds": 60,
    "maxWallMinutes": 20, "seed": 17, "pageSize": 2,
    "scenario": "incomplete-information",
    "endpoints": [{"url": "http://127.0.0.1:8000", "model": MODEL,
                   "revision": runtime["revision"], "thinking": False,
                   "temperature": 0.7, "topP": 0.8}]
}
config_path = runtime_dir / "pilot.json"
config_path.write_text(json.dumps(config, indent=2))
result = subprocess.run(["node", "lab/runner.mjs", str(config_path)], text=True, capture_output=True, timeout=25 * 60)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Inspect the saved checkpoint and logs before resuming."
outcome = json.loads(result.stdout)
print("Stop reason:", outcome["stopReason"])


A budget or inference stop is an incomplete run, not a successful pilot. The runner prints a checkpoint path. To resume after resolving the cause, copy that path into the same configuration as checkpoint, keep the other experimental settings unchanged, and run the command below. Already acknowledged product commands keep their original receipt. A new seed or model configuration belongs in a new run.

In [ ]:
# Run only when you intend to resume this same run:
# config["checkpoint"] = outcome["checkpoint"]
# config_path.write_text(json.dumps(config, indent=2))
# subprocess.run(["node", "lab/runner.mjs", str(config_path), "--resume"], check=True)


In [ ]:
run_dir = Path(outcome["checkpoint"]).parent
export = run_dir / "run.json"
subprocess.check_call(["node", "lab/report.mjs", str(export)])
subprocess.check_call(["node", "tools/rehearsal.mjs", "replay", str(export)])
shutil.copy2(lock_path, run_dir / "runtime-lock.json")
shutil.copy2(runtime_dir / "pip-freeze.txt", run_dir / "pip-freeze.txt")
(run_dir / "gpu-after.txt").write_text(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu", "--format=csv"], text=True))
# Export evidence only. Account sessions, operator keys and live checkpoints stay local.
download = Path("/content/mihenk-evidence.zip")
with zipfile.ZipFile(download, "w", zipfile.ZIP_DEFLATED) as bundle:
    for file in run_dir.rglob("*"):
        if file.is_file() and (file.name in ["run.json", "run.report.json", "run.report.md", "runtime-lock.json", "pip-freeze.txt", "gpu-after.txt"] or file.suffix == ".png" or file.name.endswith(".events.jsonl")):
            bundle.write(file, file.relative_to(run_dir))
files.download(str(download))


In [ ]:
# Stop these two notebook-owned processes when finished.
for process in [application, inference]:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
app_log.close()
model_log.close()
